In [15]:
from pathlib import Path
import sys

def find_project_root(marker="README.md"):
    current = Path.cwd().resolve()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find project root with marker: {marker}")

PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/Local/REPOSITORIES/injury_risk_classifier


In [16]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

from src.cleaning_and_features import explore_data

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
df = pd.read_csv("../injury_risk_classifier/data/interim/traffic_clean.csv")
df.head()

,top_traffic_accident_offense,first_occurrence_date,incident_address,lon,lat,tu1_travel_direction,tu2_travel_direction,road_description,road_condition,light_condition,datetime,date,time,injured,tu1_vehicle_type_binned,tu2_vehicle_type_binned,tu1_driver_action_binned,tu2_driver_action_binned,tu1_human_factor_binned,tu2_human_factor_binned
0,TRAF - ACCIDENT,2.459922e+06,I25 HWYNB / W 6TH AVE,-105.013229,39.725677,north,north,non-intersection,dry,daylight,2022-12-08 14:05:59.999991956,2022-12-08,14:05:59,0,pickup_or_utility_van,bus,no_action,no_action,no_apparent,no_apparent
1,TRAF - ACCIDENT,2.459925e+06,E 17TH AVE / N PENNSYLVANIA ST,-104.981084,39.743271,south,east,at intersection,dry,daylight,2022-12-11 11:39:59.999991056,2022-12-11,11:39:59,0,passenger_car_or_van,suv,failure_to_yield,no_action,no_apparent,no_apparent
2,TRAF - ACCIDENT,2.459925e+06,I25 HWYSB / PARK AVEW,-104.994972,39.768330,south,south,non-intersection,dry,daylight,2022-12-11 12:10:00.000004471,2022-12-11,12:10:00,0,passenger_car_or_van,passenger_car_or_van,aggressive_or_careless,no_action,no_apparent,no_apparent
3,TRAF - ACCIDENT - POLICE,2.459922e+06,I25 HWYNB / 20TH ST,-105.004334,39.760949,south,south,non-intersection,dry,dark-lighted,2022-12-08 18:26:59.999983903,2022-12-08,18:26:59,0,pickup_or_utility_van,passenger_car_or_van,aggressive_or_careless,no_action,aggressive_or_evading,no_apparent
4,TRAF - ACCIDENT - HIT & RUN,2.459922e+06,W 37TH AVE / N PECOS ST,-105.006450,39.768054,NaN,NaN,non-intersection,dry,dark-lighted,2022-12-08 19:30:00.000000000,2022-12-08,19:30:00,0,unknown,passenger_car_or_van,unknown,unknown,unknown,unknown


In [18]:
df.columns

Index(['top_traffic_accident_offense', 'first_occurrence_date',
       'incident_address', 'lon', 'lat', 'tu1_travel_direction',
       'tu2_travel_direction', 'road_description', 'road_condition',
       'light_condition', 'datetime', 'date', 'time', 'injured',
       'tu1_vehicle_type_binned', 'tu2_vehicle_type_binned',
       'tu1_driver_action_binned', 'tu2_driver_action_binned',
       'tu1_human_factor_binned', 'tu2_human_factor_binned'],
      dtype='str')

In [19]:
df.shape

(282290, 20)

In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 282290 entries, 0 to 282289
Data columns (total 20 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   top_traffic_accident_offense  282290 non-null  str    
 1   first_occurrence_date         282290 non-null  float64
 2   incident_address              282290 non-null  str    
 3   lon                           271137 non-null  float64
 4   lat                           271137 non-null  float64
 5   tu1_travel_direction          269154 non-null  str    
 6   tu2_travel_direction          258989 non-null  str    
 7   road_description              276351 non-null  str    
 8   road_condition                275950 non-null  str    
 9   light_condition               275638 non-null  str    
 10  datetime                      282290 non-null  str    
 11  date                          282290 non-null  str    
 12  time                          282290 non-null  str    


In [21]:
df.head()

,top_traffic_accident_offense,first_occurrence_date,incident_address,lon,lat,tu1_travel_direction,tu2_travel_direction,road_description,road_condition,light_condition,datetime,date,time,injured,tu1_vehicle_type_binned,tu2_vehicle_type_binned,tu1_driver_action_binned,tu2_driver_action_binned,tu1_human_factor_binned,tu2_human_factor_binned
0,TRAF - ACCIDENT,2.459922e+06,I25 HWYNB / W 6TH AVE,-105.013229,39.725677,north,north,non-intersection,dry,daylight,2022-12-08 14:05:59.999991956,2022-12-08,14:05:59,0,pickup_or_utility_van,bus,no_action,no_action,no_apparent,no_apparent
1,TRAF - ACCIDENT,2.459925e+06,E 17TH AVE / N PENNSYLVANIA ST,-104.981084,39.743271,south,east,at intersection,dry,daylight,2022-12-11 11:39:59.999991056,2022-12-11,11:39:59,0,passenger_car_or_van,suv,failure_to_yield,no_action,no_apparent,no_apparent
2,TRAF - ACCIDENT,2.459925e+06,I25 HWYSB / PARK AVEW,-104.994972,39.768330,south,south,non-intersection,dry,daylight,2022-12-11 12:10:00.000004471,2022-12-11,12:10:00,0,passenger_car_or_van,passenger_car_or_van,aggressive_or_careless,no_action,no_apparent,no_apparent
3,TRAF - ACCIDENT - POLICE,2.459922e+06,I25 HWYNB / 20TH ST,-105.004334,39.760949,south,south,non-intersection,dry,dark-lighted,2022-12-08 18:26:59.999983903,2022-12-08,18:26:59,0,pickup_or_utility_van,passenger_car_or_van,aggressive_or_careless,no_action,aggressive_or_evading,no_apparent
4,TRAF - ACCIDENT - HIT & RUN,2.459922e+06,W 37TH AVE / N PECOS ST,-105.006450,39.768054,NaN,NaN,non-intersection,dry,dark-lighted,2022-12-08 19:30:00.000000000,2022-12-08,19:30:00,0,unknown,passenger_car_or_van,unknown,unknown,unknown,unknown


In [22]:
df.describe(include="all")

,top_traffic_accident_offense,first_occurrence_date,incident_address,lon,lat,tu1_travel_direction,tu2_travel_direction,road_description,road_condition,light_condition,datetime,date,time,injured,tu1_vehicle_type_binned,tu2_vehicle_type_binned,tu1_driver_action_binned,tu2_driver_action_binned,tu1_human_factor_binned,tu2_human_factor_binned
count,282290,2.822900e+05,282290,271137.000000,271137.000000,269154,258989,276351,275950,275638,282290,282290,282290,282290.000000,282290,282290,282290,282290,282290,282290
unique,6,NaN,52971,NaN,NaN,13,16,17,20,6,267803,4818,1440,NaN,12,12,10,10,12,12
top,TRAF - ACCIDENT,NaN,I25 HWYNB / W 6TH AVE,NaN,NaN,north,north,at intersection,dry,day light,2014-11-10 15:59:59.999986584,2013-11-21,17:00:00,NaN,passenger_car_or_van,passenger_car_or_van,aggressive_or_careless,no_action,no_apparent,no_apparent
freq,181867,NaN,1980,NaN,NaN,70177,66012,88836,240418,138531,8,156,1864,NaN,131995,135164,104776,153597,94848,218837
mean,NaN,2.458555e+06,NaN,-104.958342,39.732038,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.023185,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,1.369017e+03,NaN,0.069832,0.041530,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.150492,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,2.456294e+06,NaN,-105.206197,39.614406,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,2.457404e+06,NaN,-105.004766,39.705704,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,2.458407e+06,NaN,-104.977587,39.738276,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,2.459708e+06,NaN,-104.918259,39.762007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
df.isna().mean().sort_values(ascending=False).head(30)

tu2_travel_direction            0.082543
tu1_travel_direction            0.046534
lon                             0.039509
lat                             0.039509
light_condition                 0.023564
road_condition                  0.022459
road_description                0.021039
top_traffic_accident_offense    0.000000
tu1_vehicle_type_binned         0.000000
tu1_human_factor_binned         0.000000
tu2_driver_action_binned        0.000000
tu1_driver_action_binned        0.000000
tu2_vehicle_type_binned         0.000000
datetime                        0.000000
injured                         0.000000
time                            0.000000
date                            0.000000
first_occurrence_date           0.000000
incident_address                0.000000
tu2_human_factor_binned         0.000000
dtype: float64

In [24]:
df["injured"].value_counts(normalize=False)

injured
0    275745
1      6545
Name: count, dtype: int64

In [25]:
df.head(10)

,top_traffic_accident_offense,first_occurrence_date,incident_address,lon,lat,tu1_travel_direction,tu2_travel_direction,road_description,road_condition,light_condition,datetime,date,time,injured,tu1_vehicle_type_binned,tu2_vehicle_type_binned,tu1_driver_action_binned,tu2_driver_action_binned,tu1_human_factor_binned,tu2_human_factor_binned
0,TRAF - ACCIDENT,2.459922e+06,I25 HWYNB / W 6TH AVE,-105.013229,39.725677,north,north,non-intersection,dry,daylight,2022-12-08 14:05:59.999991956,2022-12-08,14:05:59,0,pickup_or_utility_van,bus,no_action,no_action,no_apparent,no_apparent
1,TRAF - ACCIDENT,2.459925e+06,E 17TH AVE / N PENNSYLVANIA ST,-104.981084,39.743271,south,east,at intersection,dry,daylight,2022-12-11 11:39:59.999991056,2022-12-11,11:39:59,0,passenger_car_or_van,suv,failure_to_yield,no_action,no_apparent,no_apparent
2,TRAF - ACCIDENT,2.459925e+06,I25 HWYSB / PARK AVEW,-104.994972,39.768330,south,south,non-intersection,dry,daylight,2022-12-11 12:10:00.000004471,2022-12-11,12:10:00,0,passenger_car_or_van,passenger_car_or_van,aggressive_or_careless,no_action,no_apparent,no_apparent
3,TRAF - ACCIDENT - POLICE,2.459922e+06,I25 HWYNB / 20TH ST,-105.004334,39.760949,south,south,non-intersection,dry,dark-lighted,2022-12-08 18:26:59.999983903,2022-12-08,18:26:59,0,pickup_or_utility_van,passenger_car_or_van,aggressive_or_careless,no_action,aggressive_or_evading,no_apparent
4,TRAF - ACCIDENT - HIT & RUN,2.459922e+06,W 37TH AVE / N PECOS ST,-105.006450,39.768054,NaN,NaN,non-intersection,dry,dark-lighted,2022-12-08 19:30:00.000000000,2022-12-08,19:30:00,0,unknown,passenger_car_or_van,unknown,unknown,unknown,unknown
5,TRAF - ACCIDENT - HIT & RUN,2.459916e+06,S OSCEOLA ST / W BAYAUD AVE,-105.038499,39.714959,south,north,non-intersection,dry,dark-lighted,2022-12-02 02:00:00.000013415,2022-12-02,02:00:00,0,passenger_car_or_van,passenger_car_or_van,aggressive_or_careless,no_action,unknown,no_apparent
6,TRAF - ACCIDENT - HIT & RUN,2.459916e+06,300 BLOCK E 17TH AVE,-104.983103,39.743008,NaN,northeast,non-intersection,dry,daylight,2022-12-02 12:20:00.000008943,2022-12-02,12:20:00,0,unknown,passenger_car_or_van,unknown,no_action,unknown,no_apparent
7,TRAF - ACCIDENT - HIT & RUN,2.459916e+06,N OGDEN ST / E 11TH AVE,-104.975278,39.733689,south,east,at intersection,dry,dark-lighted,2022-12-02 15:59:59.999986584,2022-12-02,15:59:59,0,pickup_or_utility_van,passenger_car_or_van,failure_to_obey,no_action,no_apparent,no_apparent
8,TRAF - ACCIDENT,2.459916e+06,N CENTRAL PARK BLVD / N VERBENA ST,-104.889926,39.751852,south,south,intersection related,dry,daylight,2022-12-02 15:45:00.000000000,2022-12-02,15:45:00,0,passenger_car_or_van,passenger_car_or_van,aggressive_or_careless,lane_or_position,aggressive_or_evading,aggressive_or_evading
9,TRAF - ACCIDENT - HIT & RUN,2.459916e+06,I70 HWYWB / N NORTHFIELD QUEBEC ST,-104.903436,39.778421,north,north,intersection related,dry,dark-lighted,2022-12-02 19:29:00.000003572,2022-12-02,19:29:00,0,passenger_car_or_van,passenger_car_or_van,lane_or_position,no_action,unknown,no_apparent


In [26]:
df.isna().sum()

top_traffic_accident_offense        0
first_occurrence_date               0
incident_address                    0
lon                             11153
lat                             11153
tu1_travel_direction            13136
tu2_travel_direction            23301
road_description                 5939
road_condition                   6340
light_condition                  6652
datetime                            0
date                                0
time                                0
injured                             0
tu1_vehicle_type_binned             0
tu2_vehicle_type_binned             0
tu1_driver_action_binned            0
tu2_driver_action_binned            0
tu1_human_factor_binned             0
tu2_human_factor_binned             0
dtype: int64